In [ ]:
# %% Environment & Repository Cloning
# IMPORTANT: Replace 'https://github.com/nithin12342/company-cad-proposal.git' with your actual repository URL.
!git clone https://github.com/nithin12342/company-cad-proposal.git
%cd company-cad-proposal

# Mount Google Drive and setup model cache
from google.colab import drive
drive.mount('/content/drive')
import os
doctr_cache_dir = '/content/drive/MyDrive/doctr_models'
os.makedirs(doctr_cache_dir, exist_ok=True)
os.environ['HF_HOME'] = doctr_cache_dir
os.environ['HF_CACHE_DIR'] = doctr_cache_dir
print(f'Doctr models will be cached in: {doctr_cache_dir}')

# Install system dependencies
!apt-get install -y poppler-utils

# Install Python packages
!pip install -q torch torchvision opencv-python python-doctr matplotlib pydantic pdf2image google-genai
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

# Pre-download DocTR models to Drive
print('Downloading DocTR models to Drive...')
from doctr import models
fast_predictor = models.ocr_predictor(pretrained=True)
print('DocTR models downloaded successfully')

print('Environment setup complete')

Cloning into 'company-cad-proposal'...
remote: Enumerating objects: 247, done.
remote: Counting objects: 100% (247/247), done.
remote: Compressing objects: 100% (178/178), done.
remote: Total 247 (delta 124), reused 175 (delta 61), pack-reused 0 (from 0)
Receiving objects: 100% (247/247), 31.09 MiB | 25.86 MiB/s, done.
Resolving deltas: 100% (124/124), done.
/content/company-cad-proposal
Mounted at /content/drive
Doctr models will be cached in: /content/drive/MyDrive/doctr_models
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 0s (625 kB/s)
Selecting previously unselected package poppler

In [ ]:
# === Persistent Model Caching (Google Drive) ===
from google.colab import drive
import os

print("Mounting Google Drive for persistent model storage...")
drive.mount('/content/drive')

# Define paths
drive_models_dir = '/content/drive/MyDrive/CAD_Models'
local_models_dir = '/content/company-cad-proposal/models'  # Adjust if repo name differs
sam_checkpoint = "sam_vit_h_4b8939.pth"
drive_path = os.path.join(drive_models_dir, sam_checkpoint)
local_path = os.path.join(local_models_dir, sam_checkpoint)

# Ensure directories exist
os.makedirs(drive_models_dir, exist_ok=True)
os.makedirs(local_models_dir, exist_ok=True)

# Cache Logic
if os.path.exists(drive_path):
    print(f"✅ Found SAM weights in Google Drive: {drive_path}")
    if not os.path.exists(local_path):
        os.symlink(drive_path, local_path)
        print(f"🔗 Symlinked Drive weights to local repository: {local_path}")
    else:
        print("🔗 Symlink already exists.")
else:
    print(f"⬇️ SAM weights not found in Drive. Downloading to {drive_path} (This only happens once)...")
    !wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth -O \"{drive_path}\"
    os.symlink(drive_path, local_path)
    print("✅ Download and symlink complete!")


In [ ]:
# Locate drawing.pdf
import os
pdf_path = "/content/drive/MyDrive/company cad project/input/drawings.pdf" # Update path as needed
if not os.path.exists(pdf_path):
    raise FileNotFoundError(f"PDF not found at {pdf_path}. Please ensure the PDF is uploaded to your Drive.")
print(f"Found drawing.pdf at: {pdf_path}")
input_pdf = pdf_path

In [ ]:
# 1. Fix the typo in the source file
import os
layout_file = 'src/nodes/layout.py'
if os.path.exists(layout_file):
    with open(layout_file, 'r') as f:
        content = f.read()
    # The error shows it was looking for recognition.zpredictor
    # We replace it with the correct models.ocr_predictor
    new_content = content.replace('models.recognition.zpredictor', 'models.ocr_predictor')
    with open(layout_file, 'w') as f:
        f.write(new_content)
    print(f"Fixed typo in {layout_file}")

# 2. Setup path and reload modules to ensure changes are picked up
import sys
import importlib
sys.path.insert(0, '.')

# Import the nodes
import src.nodes.triage
import src.nodes.vectorize
import src.nodes.layout
import src.nodes.dhmot

# Force reload the modified layout module
importlib.reload(src.nodes.layout)

from src.nodes.triage import PixelTriageNode
from src.nodes.vectorize import GeometricExtractionNode
from src.nodes.layout import LayoutExtractionNode
from src.nodes.dhmot import DHMoTNode

# 3. Initialize and Execute
triage_node = PixelTriageNode(node_id='node_01_triage')
vectorize_node = GeometricExtractionNode(node_id='node_02_vectorize')
layout_node = LayoutExtractionNode(node_id='node_03_layout')
dhmot_node = DHMoTNode(node_id='node_04_dhmot')

print('Running pipeline...')
triage_output, _ = triage_node.execute(input_pdf)
if triage_output.success:
    geometry_mask_path = triage_output.data.geometry_mask_path
    text_mask_path = triage_output.data.text_mask_path
    table_mask_path = triage_output.data.table_mask_path

    vectorize_output, _ = vectorize_node.execute(geometry_mask_path, page_number=1)
    geometry = vectorize_output.data

    layout_output, _ = layout_node.execute(table_mask_path, text_mask_path, page_number=1, original_file_path=input_pdf)
    tables = layout_output.data

    dhmot_output, _ = dhmot_node.execute(geometry, tables, original_img_path=geometry_mask_path)
    if dhmot_output.success:
        axiom_manifest = dhmot_output.data
        validation_overlay_path = dhmot_output.metadata.get('validation_overlay_path')
        print('Pipeline successful')
    else:
        print('DHMoT failed')
else:
    print('Triage failed')

In [ ]:
# %%
# Display Deterministic Outputs and Download Files
import json
from IPython.display import Image, display
from google.colab import files
import os

# Print Axiom Manifest as formatted JSON
print('\n=== AXIOM MANIFEST (Pre-LLM Output) ===')
print(json.dumps(axiom_manifest, indent=4, default=str))

# Save Axiom Manifest to file for download
manifest_json = json.dumps(axiom_manifest, indent=4, default=str)
with open('axiom_manifest.json', 'w') as f:
    f.write(manifest_json)

# Display validation overlay if available
if validation_overlay_path and os.path.exists(validation_overlay_path):
    print('\n=== VALIDATION OVERLAY ===')
    try:
        display(Image(filename=validation_overlay_path))
    except Exception as e:
        print(f'Could not load overlay image: {e}')
else:
    print('\nValidation overlay path not found or file does not exist')

# Download files
print('\n=== DOWNLOADING OUTPUTS ===')
try:
    files.download('axiom_manifest.json')
    print('Downloaded: axiom_manifest.json')
except Exception as e:
    print(f'Error downloading axiom_manifest.json: {e}')

if validation_overlay_path and os.path.exists(validation_overlay_path):
    try:
        files.download(validation_overlay_path)
        print(f'Downloaded: {os.path.basename(validation_overlay_path)}')
    except Exception as e:
        print(f'Error downloading validation overlay: {e}')
else:
    print('Validation overlay not available for download.')

print('\nIsolated pre-LLM test completed successfully. Outputs are ready for download.')